# Exp 16: Downsampling

Card: [exp16_downsampling.md](../../docs/experiments/exp16_downsampling.md)


In [ ]:
import dataclasses
import matplotlib.pyplot as plt

from core.training.config import ROOT
from core.training.config import VIP
from core.training.cv import CVResult
from core.analysis.gradcam import GradCamFigures
from core.analysis.plots import pr_curves, save_figures, use_style

N_POSITIVES = None  # all labeled positives
CONFIG_DIR = ROOT / "configs" / "models"
CONFIG = CONFIG_DIR / "exp16_downsampling" / "resnet18.yaml"
# trained in exp11 on folds 0-4, only read; compared on same folds as CONFIG
EXP11 = CONFIG_DIR / "exp11_resnet18_5fold" / "resnet18.yaml"
FIGURES = ROOT / "docs" / "figures"
PREFIX = "exp16_downsampling"

# dataviz reference palette, light mode
SERIES = {"not downsampled (exp11)": "#eb6834", "downsampled": "#2a78d6"}
use_style()

## Training


In [ ]:
base = CVResult.load(EXP11)
# exp11 result holds folds 0-4: keep folds of this experiment
results = {
    "not downsampled (exp11)": dataclasses.replace(
        base, oof=base.oof[base.oof["fold"].isin(CVResult.load(CONFIG).folds)]
    )
}

In [ ]:
results["downsampled"] = CVResult.load(CONFIG)

## Precision-recall curves


In [ ]:
fig = pr_curves(results, colors=SERIES)
figures = {f"{PREFIX}_pr_curves.png": fig}
plt.show()

## Grad-CAM VIP: unseen vs trained images


In [ ]:
cams = GradCamFigures(CONFIG)
for cls in VIP:
    for train in (False, True):
        fig = cams.positives(cls, train=train, n=N_POSITIVES)
        split = "train" if train else "validation"
        figures[f"{PREFIX}_grad_cam_{cls.lower()}_{split}.png"] = fig
plt.show()

In [ ]:
save_figures(figures, FIGURES)